# Support Integrity Auditor — Severity Mismatch Classifier (Prototype 2)

**Model:** fine-tuned `microsoft/deberta-v3-small` (a full fine-tune of a small encoder).

This satisfies the requirement of using a *fine-tuned / adapter-trained* model rather than a
frozen zero-shot pipeline: every backbone weight is updated on our labelled data.

Task: binary classification — does a ticket's stated `Priority_Level` mismatch the
severity our pseudo-labelling pipeline inferred (`Is_Mismatch` = 0/1).

In [1]:
import os
# Set before torch/CUDA init: reduces VRAM fragmentation on the 4GB GPU.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

/home/aryansharma/Desktop/Aryan Pendrive Data/Support-Integrity-Auditor/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
Device: NVIDIA GeForce RTX 3050 Laptop GPU


## 1. Load data & build the input text

In [2]:
df = pd.read_csv("../dataset/preprocessed_data.csv")
print(f"Loaded {len(df)} rows")

def to_text(row):
    return (
        f"{row['Ticket_Subject']}: {row['Ticket_Description']} "
    )

df["TXT"] = df.apply(to_text, axis=1)
dataset = df[["TXT", "Final_severity_label"]].rename(columns={"Final_severity_label": "labels"})
dataset.labels = dataset.labels.map({"LOW":0,"MEDIUM":1,"HIGH":2,"CRITICAL":3})
print("\nLabel balance:")
print(dataset["labels"].value_counts(normalize=True).round(3))
dataset.head()

Loaded 20000 rows

Label balance:
labels
0    0.389
2    0.347
1    0.189
3    0.074
Name: proportion, dtype: float64


,TXT,labels
0,"Hours of operation - Individual: Hi Support, W...",0
1,"Data not syncing - Card: Hi Support, The appli...",1
2,"2FA issues - Question: Hi Support, How do I up...",2
3,"Login failed - Let: Hi Support, The dashboard ...",1
4,"Refund status - Attention: Hi Support, I have ...",2


## 2. Train / eval split

In [3]:
train_df, eval_df = train_test_split(
    dataset, test_size=0.2, random_state=42, stratify=dataset["labels"]
)
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
eval_dataset  = Dataset.from_pandas(eval_df,  preserve_index=False)
print(len(train_dataset), "train /", len(eval_dataset), "eval")

16000 train / 4000 eval


## 3. Tokenizer & model — `deberta-v3-small`

In [4]:
model_name = "microsoft/deberta-v3-small"

# DeBERTa-v3 uses a SentencePiece tokenizer (needs the `sentencepiece` package, already installed).
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
)
# Force fp32 weights BEFORE training. The checkpoint ships in fp16, and fp16=True mixed
# precision on fp16 master weights raises "Attempting to unscale FP16 gradients".
# Mixed precision needs fp32 master weights + an autocast fp16 forward pass.
model = model.float().to("cuda")

n_total = sum(p.numel() for p in model.parameters())
print(f"{model_name}: {n_total/1e6:.1f}M params (all trainable - full fine-tune)")

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 29852.70it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING  

microsoft/deberta-v3-small: 141.9M params (all trainable - full fine-tune)


In [5]:
def tokenize_function(batch):
    return tokenizer(batch["TXT"], truncation=True, max_length=128)  # tickets are ~55 tokens

tokenized_train = train_dataset.map(tokenize_function, batched=True).remove_columns(["TXT"])
tokenized_eval  = eval_dataset.map(tokenize_function,  batched=True).remove_columns(["TXT"])

tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_eval.set_format("torch",  columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)  # dynamic padding per batch

Map: 100%|██████████| 4000/4000 [00:00<00:00, 36516.35 examples/s]


## 4. Metrics

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    # 4-class severity task (LOW/MEDIUM/HIGH/CRITICAL). Every value must be a
    # native Python type: numpy scalars/arrays are not JSON serializable and
    # land in TrainerState.log_history, which breaks save_to_json at checkpoint
    # time ("Object of type ndarray is not JSON serializable").
    accuracy = float(accuracy_score(labels, predictions))
    precision, recall, f1_weighted, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )
    f1_macro = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )[2]

    return {
        "accuracy": accuracy,
        "f1_weighted": float(f1_weighted),
        "f1_macro": float(f1_macro),
        "precision": float(precision),
        "recall": float(recall),
    }

## 5. Training arguments & Trainer

In [10]:
training_args = TrainingArguments(
    output_dir="./deberta_mismatch",
    num_train_epochs=3,
    per_device_train_batch_size=4,       # 4GB GPU: 8 + accumulation keeps peak ~2.3GB
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,       # effective batch = 16
    learning_rate=2e-5,                  # standard full fine-tune LR for encoders
    warmup_steps=100,
    weight_decay=0.01,
    optim="adamw_bnb_8bit",              # 8-bit Adam: ~halves optimizer-state VRAM vs fp32 Adam
    logging_dir="./logs",
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    greater_is_better=True,              # higher F1 is better
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## 6. Train

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro,Precision,Recall
1,0.655522,0.292626,0.931250,0.931095,0.926032,0.931746,0.931250
2,0.342298,0.227935,0.936250,0.936381,0.930752,0.937286,0.936250
3,0.456401,0.220677,0.939250,0.939449,0.933272,0.940266,0.939250


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]


TrainOutput(global_step=6000, training_loss=0.5276617023547491, metrics={'train_runtime': 3787.4738, 'train_samples_per_second': 12.673, 'train_steps_per_second': 1.584, 'total_flos': 376239406111200.0, 'train_loss': 0.5276617023547491, 'epoch': 3.0})

## 7. Evaluate — full classification report & confusion matrix

In [13]:
pred = trainer.predict(tokenized_eval)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

print("Eval metrics:", {k: round(v, 4) for k, v in pred.metrics.items() if k.startswith("test_")})
print("\nClassification report:")
print(classification_report(y_true, y_pred, digits=4))
print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(y_true, y_pred))

Eval metrics: {'test_loss': 0.2926, 'test_accuracy': 0.9313, 'test_f1_weighted': 0.9311, 'test_f1_macro': 0.926, 'test_precision': 0.9317, 'test_recall': 0.9313, 'test_runtime': 35.4318, 'test_samples_per_second': 112.893, 'test_steps_per_second': 28.223}

Classification report:
              precision    recall  f1-score   support

           0     0.9389    0.9474    0.9431      1558
           1     0.9293    0.9723    0.9503       757
           2     0.9393    0.8912    0.9146      1388
           3     0.8652    0.9293    0.8961       297

    accuracy                         0.9313      4000
   macro avg     0.9182    0.9350    0.9260      4000
weighted avg     0.9317    0.9313    0.9311      4000

Confusion matrix [rows=true, cols=pred]:
[[1476   12   70    0]
 [   5  736    9    7]
 [  85   30 1237   36]
 [   6   14    1  276]]


In [14]:
e = df.iloc[eval_df.index]
e["Priority_Level"] = e["Priority_Level"].map({"Low":0,"Medium":1,"High":2,"Critical":3})
s = (y_pred != e["Priority_Level"]).astype(int)
yt = e["Is_Mismatch"]
print("\nClassification report:")
print(classification_report(yt, s, digits=4))
print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(yt, s))


Classification report:
              precision    recall  f1-score   support

           0     0.9352    0.9398    0.9375      1412
           1     0.9671    0.9645    0.9658      2588

    accuracy                         0.9557      4000
   macro avg     0.9511    0.9521    0.9516      4000
weighted avg     0.9558    0.9557    0.9558      4000

Confusion matrix [rows=true, cols=pred]:
[[1327   85]
 [  92 2496]]


> **Caveat:** `Is_Mismatch` is a *pseudo-label* derived from our own scoring heuristic.
> A high F1 means the model has learned to reproduce that heuristic — it is not a ground-truth
> accuracy measure. Treat it as "did the adapter learn the signal," not "is the rule correct."

## 8. Save the fine-tuned model

In [15]:
save_dir = "./deberta_mismatch/best"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to", save_dir)

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.05s/it]

Saved to ./deberta_mismatch/best
